In [2]:
import os, io, zipfile, datetime, urllib.request
import numpy as np, pandas as pd, tldextract
import features as F

os.makedirs("../data/zeroday", exist_ok=True)
today = datetime.date.today().isoformat()

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (student project)"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return r.read()

# Everything PhiUSIIL already contains, across all three splits
seen = pd.concat([pd.read_csv(f"../data/splits/{s}.csv") for s in ["train", "val", "test"]])
seen_hosts = set(seen["URL"].map(lambda u: F.to_text(u, "host")))
seen_reg   = set(seen["URL"].map(lambda u: tldextract.extract(u).registered_domain))

# --- Phishing: OpenPhish. Rerun this part once a day; new hosts are appended. ---
raw = fetch("https://openphish.com/feed.txt").decode("utf-8", "ignore").split()
ph = pd.DataFrame({"URL": raw})
ph["host"] = ph["URL"].map(lambda u: F.to_text(u, "host"))
ph = ph.drop_duplicates("host")
ph = ph[~ph["host"].isin(seen_hosts)].assign(collected=today)
p_path = "../data/zeroday/phish.csv"
if os.path.exists(p_path):
    ph = pd.concat([pd.read_csv(p_path), ph]).drop_duplicates("host", keep="first")
ph.to_csv(p_path, index=False)

# --- Legitimate: Tranco. Runs only once. ---
l_path = "../data/zeroday/legit.csv"
if not os.path.exists(l_path):
    z = zipfile.ZipFile(io.BytesIO(fetch("https://tranco-list.eu/top-1m.csv.zip")))
    tr = pd.read_csv(z.open(z.namelist()[0]), header=None, names=["rank", "domain"])
    tr = tr[tr["rank"] > 10_000]                      # skip the giants PhiUSIIL is full of
    tr["reg"] = tr["domain"].map(lambda d: tldextract.extract(d).registered_domain)
    leg = tr[~tr["reg"].isin(seen_reg)].sample(2000, random_state=42)
    leg["host"] = leg["domain"].map(lambda d: F.to_text(d, "host"))
    leg.assign(collected=today).to_csv(l_path, index=False)

leg = pd.read_csv(l_path)
print(f"phishing hosts: {len(ph):,}   legitimate hosts: {len(leg):,}")
print(ph["host"].sample(min(15, len(ph)), random_state=0).to_string(index=False))

/var/folders/wj/51gfcw215k91r2mb2mnm9gjw0000gn/T/ipykernel_48348/2304232051.py:16: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  seen_reg   = set(seen["URL"].map(lambda u: tldextract.extract(u).registered_domain))
/var/folders/wj/51gfcw215k91r2mb2mnm9gjw0000gn/T/ipykernel_48348/2304232051.py:35: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  tr["reg"] = tr["domain"].map(lambda d: tldextract.extract(d).registered_domain)


phishing hosts: 256   legitimate hosts: 2,000
                   porfolio-099.pages.dev
        android.webwindos-whatsapp.com.cn
               abonnement-resiliation.net
               docs.web86-whatsapp.com.cn
          solicitud-gamer-i1cg.vercel.app
                      tetes-air.pages.dev
realscout.sfo3.cdn.digitaloceanspaces.com
                       infora.duckdns.org
           docs.webwindos-whatsapp.com.cn
                  whatsapp.checkleaked.cc
                   pay-network.vercel.app
                 naimishgupta86.github.io
                   webwin-whatsapp.com.cn
                     bancosentander.us.cc
                       krast-2.vercel.app


In [4]:
import pandas as pd, tldextract
import features as F

icann = tldextract.TLDExtract(include_psl_private_domains=False)
priv  = tldextract.TLDExtract(include_psl_private_domains=True)   # knows vercel.app, github.io, ...

def annotate(df):
    df = df.copy()
    df["owner"]    = df["host"].map(lambda h: priv(h).top_domain_under_public_suffix)
    df["platform"] = df["host"].map(lambda h: priv(h).suffix != icann(h).suffix)
    return df

ph  = annotate(pd.read_csv("../data/zeroday/phish.csv"))
leg = annotate(pd.read_csv("../data/zeroday/legit.csv"))
print(f"phishing: {len(ph)} hosts from {ph['owner'].nunique()} distinct owners")
print(f"platform-hosted share   phishing {ph['platform'].mean():.1%}   legit {leg['platform'].mean():.1%}")
print("\nTop platforms in the phishing set:")
print(ph.loc[ph["platform"], "host"].map(lambda h: priv(h).suffix).value_counts().head(10).to_string())

tr = pd.read_csv("../data/splits/train.csv")
tr["host"] = tr["URL"].map(lambda u: F.to_text(u, "host"))
tr = annotate(tr)
print("\nPhiUSIIL train, platform-hosted share by label (1 = phishing):")
print(tr.groupby("label")["platform"].mean().round(4).to_string())

phishing: 256 hosts from 178 distinct owners
platform-hosted share   phishing 38.7%   legit 0.1%

Top platforms in the phishing set:
host
pages.dev       28
github.io       15
vercel.app      14
blogspot.com     7
workers.dev      6
gitbook.io       4
replit.app       3
webflow.io       3
azurefd.net      3
netlify.app      3

PhiUSIIL train, platform-hosted share by label (1 = phishing):
label
0    0.0008
1    0.2601


In [5]:
import io, zipfile, datetime, urllib.request
import pandas as pd
import features as F

today = datetime.date.today().isoformat()

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (student project)"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return r.read()

z = zipfile.ZipFile(io.BytesIO(fetch("https://tranco-list.eu/top-1m.csv.zip")))
tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=["rank", "domain"])

# Keep only platform subdomains (takes a minute over 1M rows)
is_plat = tranco["domain"].map(lambda d: priv(d).suffix != icann(d).suffix)
lp = tranco[is_plat].copy()
lp["host"] = lp["domain"].map(lambda d: F.to_text(d, "host"))
lp = annotate(lp)
lp = lp[lp["owner"].fillna("") != ""]                  # drops bare "github.io" itself

# Nothing PhiUSIIL contains, nothing in the phishing set
seen = pd.concat([pd.read_csv(f"../data/splits/{s}.csv") for s in ["train", "val", "test"]])
seen_hosts = set(seen["URL"].map(lambda u: F.to_text(u, "host")))
lp = lp[~lp["host"].isin(seen_hosts) & ~lp["host"].isin(set(ph["host"]))]

lp.assign(collected=today).to_csv("../data/zeroday/legit_platform.csv", index=False)
print(f"legitimate platform-hosted hosts: {len(lp):,}")
print(lp["host"].map(lambda h: priv(h).suffix).value_counts().head(15).to_string())

legitimate platform-hosted hosts: 0
Series([], )


In [6]:
import io, zipfile, datetime, urllib.request
import pandas as pd
import features as F

today = datetime.date.today().isoformat()

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (student project)"})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()

z = zipfile.ZipFile(io.BytesIO(fetch("http://s3-us-west-1.amazonaws.com/umbrella-static/top-1m.csv.zip")))
umb = pd.read_csv(z.open(z.namelist()[0]), header=None, names=["rank", "domain"])
umb["domain"] = umb["domain"].astype(str).str.lower().str.strip(".")
print(f"Umbrella rows: {len(umb):,}   sample: {umb['domain'].head(5).tolist()}")

# Platform subdomains only (a minute or two over 1M rows)
is_plat = umb["domain"].map(lambda d: priv(d).suffix != icann(d).suffix)
lp = umb[is_plat].copy()
lp["host"] = lp["domain"].map(lambda d: F.to_text(d, "host"))
lp = annotate(lp)
lp = lp[lp["owner"].fillna("") != ""]                 # drop bare platform domains
lp = lp.sort_values("rank").drop_duplicates("owner")  # one host per owner, most popular kept

seen = pd.concat([pd.read_csv(f"../data/splits/{s}.csv") for s in ["train", "val", "test"]])
seen_hosts = set(seen["URL"].map(lambda u: F.to_text(u, "host")))
lp = lp[~lp["host"].isin(seen_hosts) & ~lp["owner"].isin(set(ph["owner"]))]

lp.assign(source="umbrella", collected=today).to_csv("../data/zeroday/legit_platform.csv", index=False)
print(f"legitimate platform-hosted hosts: {len(lp):,}")
print(lp["host"].map(lambda h: priv(h).suffix).value_counts().head(15).to_string())

Umbrella rows: 1,000,000   sample: ['google.com', 'gstatic.com', 'www.google.com', 'microsoft.com', 'ezviz7.com']
legitimate platform-hosted hosts: 32,386
host
cloudfront.net                         11337
trafficmanager.net                      1980
s3.amazonaws.com                        1361
us-east-1.elb.amazonaws.com             1262
blob.core.windows.net                   1248
akamaized.net                           1023
servicebus.windows.net                   779
googleapis.com                           547
us-west-2.elb.amazonaws.com              512
rsc.cdn77.org                            475
azurewebsites.net                        471
h5games.usercontent.goog                 464
execute-api.us-east-1.amazonaws.com      405
azureedge.net                            361
edgekey.net                              339


In [7]:
plat = lambda h: priv(h).suffix

lp_all = pd.read_csv("../data/zeroday/legit_platform.csv")
lp_all["platform_suffix"] = lp_all["host"].map(plat)

ph_plat = ph.loc[ph["platform"], "host"].map(plat)
target  = set(ph_plat)

lpm = lp_all[lp_all["platform_suffix"].isin(target)].copy()
lpm.to_csv("../data/zeroday/legit_platform_matched.csv", index=False)

cmp = pd.DataFrame({"phishing": ph_plat.value_counts(),
                    "legit": lpm["platform_suffix"].value_counts()}).fillna(0).astype(int)
cmp = cmp.loc[sorted(target, key=lambda s: -cmp.loc[s, "phishing"])]
print(f"matched legitimate platform hosts: {len(lpm):,} across {lpm['platform_suffix'].nunique()} platforms")
print(cmp.to_string())

matched legitimate platform hosts: 2,258 across 17 platforms
                                  phishing  legit
pages.dev                               28    163
github.io                               15    224
vercel.app                              14     52
blogspot.com                             7    117
workers.dev                              6    194
gitbook.io                               4      1
webflow.io                               3      0
netlify.app                              3     36
azurefd.net                              3     58
replit.app                               3      4
framer.ai                                2      0
northeurope-01.azurewebsites.net         2      1
duckdns.org                              2     16
onrender.com                             1     18
typedream.app                            1      0
us.cc                                    1      1
trycloudflare.com                        1      1
s3.amazonaws.com                       

In [8]:
import numpy as np, pandas as pd

CAP  = 200
plat = lambda h: priv(h).suffix

ph  = annotate(pd.read_csv("../data/zeroday/phish.csv"))
leg = annotate(pd.read_csv("../data/zeroday/legit.csv"))
leg = leg[~leg["platform"]]                                   # drops the 2 bare platform domains

ph["platform_suffix"] = np.where(ph["platform"], ph["host"].map(plat), "")
target = set(ph.loc[ph["platform"], "platform_suffix"])

lp_all = pd.read_csv("../data/zeroday/legit_platform.csv")
lp_all["platform_suffix"] = lp_all["host"].map(plat)
lpm = (lp_all[lp_all["platform_suffix"].isin(target)]
       .sample(frac=1, random_state=42).groupby("platform_suffix").head(CAP))

cols = ["host", "label", "slice", "platform_suffix", "owner", "source", "collected"]
zd = pd.concat([
    ph.assign(label=1, slice=np.where(ph["platform"], "platform", "own"), source="openphish")[cols],
    leg.assign(label=0, slice="own", platform_suffix="", source="tranco")[cols],
    lpm.assign(label=0, slice="platform", source="umbrella")[cols],
], ignore_index=True)

clash = zd.groupby("host")["label"].nunique()
zd = zd[~zd["host"].isin(clash[clash > 1].index)].drop_duplicates("host")

zd.to_csv("../data/zeroday/zeroday_eval.csv", index=False)
print(zd.groupby(["slice", "label"]).size().unstack().rename(columns={0: "legit", 1: "phishing"}))
print(f"\ntotal {len(zd):,} hosts | phishing owners {zd.loc[zd.label == 1, 'owner'].nunique()}")

label     legit  phishing
slice                    
own        1999       157
platform   1073        99

total 3,328 hosts | phishing owners 178
